In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# HUMAN ACTIVITY RECOGNITION - SVM MODEL (WITH VISUALIZATION)

# 1. Import libraries
import pandas as pd
import numpy as np
import os
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# 2. UNZIP DATASET (if needed)

# Check if the unzipped directory exists in the current working directory.
# If not, attempt to unzip the file located in Google Drive.
if not os.path.exists("UCI HAR Dataset"):
    zip_file_path = "/content/drive/MyDrive/ML/UCI HAR Dataset.zip"
    if os.path.exists(zip_file_path):
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall() # Extracts to the current working directory (/content/)
        print("Dataset unzipped successfully.")
    else:
        print(f"Error: Zip file not found at {zip_file_path}. Please ensure it's uploaded to your Google Drive.")
else:
    print("Dataset already unzipped.")

# 3. LOAD DATA

X_train = pd.read_csv("UCI HAR Dataset/train/X_train.txt", delim_whitespace=True, header=None)
y_train = pd.read_csv("UCI HAR Dataset/train/y_train.txt", header=None)

X_test = pd.read_csv("UCI HAR Dataset/test/X_test.txt", delim_whitespace=True, header=None)
y_test = pd.read_csv("UCI HAR Dataset/test/y_test.txt", header=None)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# 2. Create Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('svm', SVC())
])

# 3. Hyperparameter Grid
param_grid = {
    'pca__n_components': [100, 120, 150],
    'svm__C': [1, 10, 50],
    'svm__gamma': [0.001, 0.01, 0.1],
    'svm__kernel': ['rbf']
}

# 4. Grid Search
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("\n✅ Best Parameters:")
print(grid.best_params_)

# 5. Predictions
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("\n✅ Accuracy:", accuracy)
print("\n📊 Best Cross-Validation Accuracy:")
print(grid.best_score_)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# 7. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.title("Confusion Matrix - SVM (Optimized)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.show()